### Middleware

Middleware provides a way to more tighty control what happens inside the agent. Middleware is useful for the following:

- Tracking the agent behaviour with loggging, analytics and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guadrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarisation is useful for the following:

- Long running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

### Message based summarisation
llm=ChatGroq(
    model="llama-3.3-70b-versatile"
)
agent =create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [13]:
### Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4 ?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}\n")
    print(f"Messages: {len(response['messages'])}\n\n")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to ask mathematical questions and receive answers.\n\n## SUMMARY\nThe user asked a series of basic mathematical questions, including addition, multiplication, division, and subtraction, and received the correct answers: 2+2=4, 10*5=50, 100/4=25, and the conversation implies more questions were asked but only these answers were summarized initially. Additional questions were posed: 15-7, 3*3, and 4*4, with answers 8, 9, and 16, respectively.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user should proceed with a new mathematical question to continue the session.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='9a4909eb-1404-4b42-bcd5-0b87e6d969c4'), AIMessage(content='100 / 4 = 25.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 238, 'total_tokens': 247, 'completion_ti

### token size


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool,gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


llm=ChatGroq(
    model="llama-3.3-70b-versatile"
)
agent =create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

#Token Counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)

In [18]:
### Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai","Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~None tokens, 3 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find suitable accommodation options in various cities, with the overall task of identifying hotels that meet their budget and preferences, currently in Dubai and Singapore.\n\n## SUMMARY\nThe conversation history includes lists of top-rated hotels in Dubai and Singapore, categorized by budget and location. The user has been provided with options ranging from luxury to budget hotels in both cities and is considering their preferences and budget. The hotels in Singapore are categorized as luxury, mid-range, budget, and hostels, with options such as Marina Bay Sands, Hotel Jen Singapore, Ibis Singapore on Bencoolen, and The POD Boutique Capsule Hostel.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next steps would be for the user to review the provided hotel options in Dubai and Singapore, consider their budget and preferences, and use onl

### Human In The loop Middleware

Pause agent execution for human approval,editing or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High stake operations require human approval(eg. database writes, financial transactions).
- Compilance Workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [46]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def read_email_tool(email_id: str) ->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str,subject: str,body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subjects {subject}"

In [47]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="llama-3.3-70b-versatile"
)

agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [48]:
from langchain_core.messages import HumanMessage

config={"configurable": {"thread_id": "test-approve"}}

#step 1: request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with 'Hello' and body 'How are you?")]},
    config=config
)

In [49]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='098fe77a-3a70-410a-8c6c-2a96d965e0ce'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bkz3qe5b5', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 308, 'total_tokens': 340, 'completion_time': 0.044777702, 'completion_tokens_details': None, 'prompt_time': 0.015911377, 'prompt_tokens_details': None, 'queue_time': 0.052712312, 'total_time': 0.060689079}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7e4d-e8b9-7f21-8d0b-fe95c947cf83-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Ho

In [68]:
from langgraph.types import Command

#Step 2: Approve

if "__interrupt__" in result:
    print("⏸️ Paused!!!!!!!!!!! Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused!!!!!!!!!!! Approving....
✅ Result: Email sent successfully. Is there anything else I can help you with?


In [51]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='098fe77a-3a70-410a-8c6c-2a96d965e0ce'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bkz3qe5b5', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 308, 'total_tokens': 340, 'completion_time': 0.044777702, 'completion_tokens_details': None, 'prompt_time': 0.015911377, 'prompt_tokens_details': None, 'queue_time': 0.052712312, 'total_time': 0.060689079}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7e4d-e8b9-7f21-8d0b-fe95c947cf83-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Ho

### Reject

In [64]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str,subject: str,body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

llm=ChatGroq( model="llama-3.3-70b-versatile" )

agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve","edit","reject"],
                },
                "read_email_tool": False
            }
        ),
    ],
)

In [65]:
config = {"configurable": {"thread_id": "test_reject"}}

#Step 1:Request
result = agent.invoke( {"messages": [HumanMessage(content="Send email to john@test.com with 'Hello' and body 'How are you?")]}, config=config )

In [66]:
from langgraph.types import Command 
#Step 2: Reject
if "__interrupt__" in result: 
    print("⏸️ Paused!!!!!!!!!!! Approving....")
    result = agent.invoke( 
        Command( 
            resume={ 
                "decisions": [ 
                    {"type": "reject"} 
                ] 
            } 
        ), config=config ) 
    print(f"✅ Result: {result}")

⏸️ Paused!!!!!!!!!!! Approving....
✅ Result: {'messages': [HumanMessage(content="Send email to john@test.com with 'Hello' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='464a397a-c0c1-4810-ba55-134750911909'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'tegta1d4a', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 308, 'total_tokens': 340, 'completion_time': 0.043759511, 'completion_tokens_details': None, 'prompt_time': 0.018637034, 'prompt_tokens_details': None, 'queue_time': 0.053216305, 'total_time': 0.062396545}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7e69-959d-7902-97ec-4bb78cdabfc6-0', tool_calls=[{'nam